# Granite 4.1 — Document Q&A (RAG)

## Imports

In [1]:
from pprint import pprint

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Tokenizer

In [2]:
MODEL_ID = "ibm-granite/granite-4.1-3b"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Architecture: ['GraniteForCausalLM']
Parameters: 3,402,836,480
Device: mps:0
Dtype: torch.bfloat16


## Define Documents

In [4]:
documents = [
    {
        "doc_id": 0,
        "title": "Mount Kilimanjaro",
        "text": (
            "Mount Kilimanjaro is a dormant volcano in Tanzania. It is the highest "
            "mountain in Africa, with a summit elevation of 5,895 metres (19,341 ft) "
            "above sea level."
        ),
    },
    {
        "doc_id": 1,
        "title": "Denali",
        "text": (
            "Denali, also known as Mount McKinley, is the highest mountain peak in "
            "North America, located in Alaska, with a summit elevation of 6,190 "
            "metres (20,310 ft)."
        ),
    },
    {
        "doc_id": 2,
        "title": "Mount Everest",
        "text": (
            "Mount Everest, located in the Mahalangur Himal sub-range of the "
            "Himalayas, is Earth's highest mountain above sea level, with a summit "
            "elevation of 8,849 metres (29,032 ft)."
        ),
    },
]

pprint(documents, sort_dicts=False, width=120)

[{'doc_id': 0,
  'title': 'Mount Kilimanjaro',
  'text': 'Mount Kilimanjaro is a dormant volcano in Tanzania. It is the highest mountain in Africa, with a summit '
          'elevation of 5,895 metres (19,341 ft) above sea level.'},
 {'doc_id': 1,
  'title': 'Denali',
  'text': 'Denali, also known as Mount McKinley, is the highest mountain peak in North America, located in Alaska, '
          'with a summit elevation of 6,190 metres (20,310 ft).'},
 {'doc_id': 2,
  'title': 'Mount Everest',
  'text': "Mount Everest, located in the Mahalangur Himal sub-range of the Himalayas, is Earth's highest mountain "
          'above sea level, with a summit elevation of 8,849 metres (29,032 ft).'}]


## Single Document Answer

In [5]:
messages = [
    {"role": "user", "content": "How tall is Mount Kilimanjaro?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following documents. You may use one or more documents to assist with the user query.

You are given a list of documents within <documents></documents> XML tags:
<documents>
{"doc_id": 0, "title": "Mount Kilimanjaro", "text": "Mount Kilimanjaro is a dormant volcano in Tanzania. It is the highest mountain in Africa, with a summit elevation of 5,895 metres (19,341 ft) above sea level."}
{"doc_id": 1, "title": "Denali", "text": "Denali, also known as Mount McKinley, is the highest mountain peak in North America, located in Alaska, with a summit elevation of 6,190 metres (20,310 ft)."}
{"doc_id": 2, "title": "Mount Everest", "text": "Mount Everest, located in the Mahalangur Himal sub-range of the Himalayas, is Earth's highest mountain above sea level, with a summit elevation of 8,849 metres (29,032 ft)."}
</documents>

Write the response to the user's input by strictly aligning with the facts in the provid

In [6]:
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=150)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.


## Multiple Document Synthesis

In [7]:
messages = [
    {"role": "user", "content": "Which is taller, Kilimanjaro or Denali?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=150)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

Denali is taller than Kilimanjaro. Denali has a summit elevation of 6,190 metres (20,310 ft), while Kilimanjaro stands at 5,895 metres (19,341 ft) above sea level.


## Unanswerable Query

In [8]:
messages = [
    {"role": "user", "content": "What is the population of Tanzania?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=150)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The provided documents do not contain information about the population of Tanzania.


## Multi-Turn with Documents

In [9]:
messages = [
    {"role": "user", "content": "How tall is Mount Kilimanjaro?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=150)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.


In [10]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': 'How tall is Mount Kilimanjaro?'},
 {'role': 'assistant', 'content': 'Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.'}]


In [12]:
prompt = "How does that compare to Everest?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': 'How tall is Mount Kilimanjaro?'},
 {'role': 'assistant', 'content': 'Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.'},
 {'role': 'user', 'content': 'How does that compare to Everest?'}]


In [13]:
inputs = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=150)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

Mount Everest is taller than Mount Kilimanjaro. Mount Everest has a summit elevation of 8,849 metres (29,032 ft), which is higher than Mount Kilimanjaro's 5,895 metres (19,341 ft).
